# 04: Performances du modèle (graphiques de présentation)

> **Prototype pédagogique. Non destiné au diagnostic.**

**Important: pas de courbe train/test ici.** MedGemma est utilisé *zero-shot*
(par prompting) : il n'est **pas entraîné** sur RSNA, il n'y a donc ni epoch, ni
courbe d'apprentissage train/validation. Les graphiques ci-dessous sont les
diagnostics d'**évaluation** classiques, calculés sur les cas RSNA tenus à part :

1. matrice de confusion ; 2. précision / rappel / F1 par classe ;
3. distribution de confiance (correct vs incorrect) ; 4. courbe risque-couverture ;
5. diagramme de fiabilité (calibration) ; 6. distribution de latence.

Données : `eval/perf/baseline_predictions.csv` (MedGemma, prompt baseline, 246 cas (run arrêté avant 300)),
produit par `python eval/run_evaluation.py --engine medgemma --mode baseline --cases data/rsna_cases_large.csv --out-dir eval/perf`.

In [1]:
import sys
from pathlib import Path

# Racine du dépôt, quel que soit le dossier de lancement du kernel.
ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src').is_dir() and (candidate / 'prompts').is_dir():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from eval import build_performance_charts as pc

# Ce notebook LIT des prédictions déjà calculées (aucun GPU requis).
# Jeu de ~300 cas si disponible, sinon repli sur le jeu de 30.
candidates = [ROOT / 'eval' / 'perf' / 'baseline_predictions.csv',
              ROOT / 'eval' / 'results' / 'baseline_predictions.csv']
pred_path = next((p for p in candidates if p.exists()), None)
if pred_path is None:
    raise FileNotFoundError(
        "Aucun fichier de prédictions trouvé. Générez-le d'abord :\n"
        "python eval/run_evaluation.py --engine medgemma --mode baseline "
        "--cases data/rsna_cases.csv --out-dir eval/perf")
rows = pc.load(pred_path)
print('source:', pred_path.relative_to(ROOT), '|', len(rows), 'cas')

source: eval\perf\baseline_predictions.csv | 246 cas


In [2]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
pc.plot_confusion(rows, axes[0, 0])
pc.plot_prf(rows, axes[0, 1])
pc.plot_conf_hist(rows, axes[0, 2])
pc.plot_risk_coverage(rows, axes[1, 0])
pc.plot_calibration(rows, axes[1, 1])
pc.plot_latency(rows, axes[1, 2])
fig.suptitle(f'MedGemma zero-shot: performances ({len(rows)} cas RSNA)', fontsize=14)
fig.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()

C:\Users\natha\AppData\Local\Temp\ipykernel_14464\3178001867.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()


## Chiffres synthétiques

In [3]:
prf = pc.per_class_prf(rows)
acc = sum(r['label'] == r['predicted_class'] for r in rows) / len(rows)
print(f'Accuracy globale : {acc:.3f}')
pd.DataFrame({c: {'précision': p, 'rappel': r, 'F1': f} for c, (p, r, f) in prf.items()}).T.round(3)

Accuracy globale : 0.923


,précision,rappel,F1
normal,0.957,0.957,0.957
suspected_opacity,1.000,0.891,0.943


## Lecture pour la soutenance

- **Matrice de confusion** : montre où MedGemma confond normal / opacité et quand il s'abstient (`uncertain`).
- **Risque-couverture** : si on n'accepte que les cas au-dessus d'un seuil de confiance, l'accuracy monte mais la couverture baisse: c'est le fondement de la règle `uncertain`.
- **Calibration** : indique si une confiance de 0,8 correspond vraiment à ~80 % de bonnes réponses. Un écart à la diagonale = confiance mal calibrée (limite documentée).
- **Latence** : coût réel sur GPU 6 Go, à mentionner comme contrainte de déploiement.

Tous les panneaux sont aussi exportés en PNG individuels via
`python eval/build_performance_charts.py` (dossier `eval/perf/charts/`).